# Multi-Scalper — Trades Replay (NB24 engine on a date range)

هدف: **همان موتور بک‌تست نوت‌بوک ۲۴** را روی یک بازه‌ی زمانی اجرا کن و **هر ترید** (entry/exit/SL/TP/PnL) را به CSV ذخیره کن — تا قابل مقایسه با ترید‌های live باشد.

## ورودی
- CSVها: `notebooks/data/<SYM>/{M5,H1,D1}/ohlcv.csv`
- کانفیگ‌ها: `notebooks/results/multi_symbol_scalper/<SYM>/config.json` (همان کانفیگی که ربات live استفاده می‌کنه)
- بازه‌ی زمانی: `BACKTEST_FROM` و `BACKTEST_TO` در سل ۲

## خروجی
- `notebooks/data/replay_trades_<FROM>_<TO>.csv` — لیست **همه‌ی ترید‌های شبیه‌سازی‌شده** در بازه
- `notebooks/data/replay_summary_<FROM>_<TO>.csv` — خلاصه‌ی per-symbol (#trades, WR, net $)

## تفاوت با NB30/NB31
- NB30: فقط لیست **سیگنال‌ها** را مقایسه می‌کند (entry-only).
- NB31: per-bar diag diff (بدون شبیه‌سازی ترید کامل).
- **این نوت‌بوک (NB33):** شبیه‌سازی کامل ترید — از ورود تا SL/TP/time-stop — تا بتوانی **خود ترید** را با live مقایسه کنی.

## ۱) Imports + پارامتر بازه‌ی بک‌تست

In [1]:
from __future__ import annotations
import json, warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Dict

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.5f}'.format)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR     = PROJECT_ROOT / 'notebooks' / 'data'
RESULTS_DIR  = PROJECT_ROOT / 'notebooks' / 'results' / 'multi_symbol_scalper'
OUT_DIR      = PROJECT_ROOT / 'notebooks' / 'data'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ============================================================================
# پارامتر‌های قابل تغییر — بازه‌ی بک‌تست
# ============================================================================
BACKTEST_FROM = '2026-05-27'              # شامل این روز
BACKTEST_TO   = '2026-05-30'              # تا قبل از این روز (exclusive)

# نمادهای basket فعلی live — همان لیست run_multi_scalper.py
BASKET = ['GBPUSD', 'XAUUSD', 'GBPCAD', 'USDMXN', 'EURJPY', 'EURCAD', 'AUDCAD']

# Warmup قبل از BACKTEST_FROM که اندیکاتورها grand-state داشته باشن (EMA, ATR, ADX, RSI ...).
# 1000 bar M5 = ~ 3.5 روز — معمولاً کافیه. اگر برخی نماد M5 ندارن، بیشتر بذار.
WARMUP_DAYS = 30

print(f'Window: {BACKTEST_FROM} → {BACKTEST_TO}  (warmup {WARMUP_DAYS} days)')
print(f'Basket: {BASKET}')

Window: 2026-05-27 → 2026-05-30  (warmup 30 days)
Basket: ['GBPUSD', 'XAUUSD', 'GBPCAD', 'USDMXN', 'EURJPY', 'EURCAD', 'AUDCAD']


## ۲) ثابت‌ها و SYMBOLS_CFG — کپی دقیق از NB24

In [2]:
# Same constants as NB24 (must match — otherwise BT diverges)
BROKER_TO_NY_H = 7

EMA_FAST       = 20
EMA_TREND_H1   = 50
EMA_TREND_D1   = 50
BB_PERIOD      = 20
BB_STD         = 2.0
RSI_PERIOD     = 14
RSI_OS         = 35.0
RSI_OB         = 65.0
ATR_PERIOD     = 14

PULLBACK_TOLERANCE_ATR = 0.4
PIN_BAR_WICK_RATIO     = 0.60

# Risk / SL
RR                  = 2.0
STRUCT_LOOKBACK_BARS = 12
SL_BUFFER_ATR       = 0.10
MAX_HOLD_BARS       = 288
ONE_TRADE_AT_A_TIME = True

# Cost model — کپی دقیق از NB24 cell 2 (Errante 2026 quote table)
SYMBOLS_CFG: Dict[str, dict] = {
    'GBPUSD':  {'pip_size': 0.0001, 'pip_value_usd_per_002lot': 0.2000,  'spread_pips':   1.6},
    'EURJPY':  {'pip_size': 0.01,   'pip_value_usd_per_002lot': 0.1256,  'spread_pips':   2.9001},
    'EURCAD':  {'pip_size': 0.0001, 'pip_value_usd_per_002lot': 0.1447,  'spread_pips':   2.8001},
    'GBPCAD':  {'pip_size': 0.0001, 'pip_value_usd_per_002lot': 0.1447,  'spread_pips':   3.0},
    'AUDCAD':  {'pip_size': 0.0001, 'pip_value_usd_per_002lot': 0.1447,  'spread_pips':   2.0},
    'USDMXN':  {'pip_size': 0.0001, 'pip_value_usd_per_002lot': 0.01155, 'spread_pips': 103.7},
    'XAUUSD':  {'pip_size': 0.01,   'pip_value_usd_per_002lot': 0.0200,  'spread_pips':  17.0},
}
print('constants set.')

constants set.


## ۳) Indicators + Features + Filters — کپی دقیق از NB24 cell 4

In [3]:
def ema(s, n):
    return s.ewm(span=n, adjust=False).mean()

def rsi(close, n=14):
    d = close.diff()
    gain = d.clip(lower=0); loss = (-d).clip(lower=0)
    ag = gain.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    al = loss.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    rs = ag / al.replace(0, np.nan)
    return 100 - 100 / (1 + rs)

def atr(df, n=14):
    tr = pd.concat([
        df['high']-df['low'],
        (df['high']-df['close'].shift()).abs(),
        (df['low'] -df['close'].shift()).abs(),
    ], axis=1).max(axis=1)
    return tr.ewm(span=n, adjust=False).mean()

def adx(df, n=14):
    up   = df['high'].diff()
    down = -df['low'].diff()
    plus_dm  = pd.Series(np.where((up>down) & (up>0), up, 0.0),   index=df.index)
    minus_dm = pd.Series(np.where((down>up) & (down>0), down, 0.0), index=df.index)
    tr = pd.concat([df['high']-df['low'],
                    (df['high']-df['close'].shift()).abs(),
                    (df['low'] -df['close'].shift()).abs()], axis=1).max(axis=1)
    atr_w = tr.ewm(alpha=1/n, adjust=False, min_periods=n).mean()
    plus_di  = 100 * plus_dm.ewm(alpha=1/n, adjust=False, min_periods=n).mean()  / atr_w.replace(0, np.nan)
    minus_di = 100 * minus_dm.ewm(alpha=1/n, adjust=False, min_periods=n).mean() / atr_w.replace(0, np.nan)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    return dx.ewm(alpha=1/n, adjust=False, min_periods=n).mean().fillna(0)

def macd_hist(close, fast=12, slow=26, signal=9):
    f = ema(close, fast); s = ema(close, slow)
    line = f - s
    sig  = ema(line, signal)
    return line - sig

def stoch(df, k=14, d=3):
    ll = df['low'].rolling(k).min()
    hh = df['high'].rolling(k).max()
    k_pct = (100 * (df['close'] - ll) / (hh - ll).replace(0, np.nan)).fillna(50)
    d_pct = k_pct.rolling(d).mean().fillna(50)
    return k_pct, d_pct

def add_m5_features(df):
    df = df.copy()
    df['ema20']  = ema(df['close'], EMA_FAST)
    df['rsi']    = rsi(df['close'], RSI_PERIOD)
    df['atr']    = atr(df, ATR_PERIOD)
    mid = df['close'].rolling(BB_PERIOD).mean()
    std = df['close'].rolling(BB_PERIOD).std()
    df['bb_mid'], df['bb_up'], df['bb_lo'] = mid, mid+BB_STD*std, mid-BB_STD*std
    df['body']  = (df['close']-df['open']).abs()
    df['range'] = (df['high']-df['low']).clip(lower=1e-9)
    df['upper_wick'] = df['high']-df[['open','close']].max(axis=1)
    df['lower_wick'] = df[['open','close']].min(axis=1)-df['low']
    df['adx']        = adx(df, 14)
    df['macd_hist']  = macd_hist(df['close'])
    df['stoch_k'], df['stoch_d'] = stoch(df, k=14, d=3)
    return df

def add_htf_trend(df, n):
    df = df.copy()
    e = ema(df['close'], n); slope = e.diff()
    df['ema_trend'] = e
    df['trend_dir'] = np.where((df['close']>e)&(slope>0), 1,
                       np.where((df['close']<e)&(slope<0), -1, 0))
    h1_atr = atr(df, 14)
    df['trend_strength'] = (e.diff(6).abs() / h1_atr.replace(0, np.nan)).fillna(0)
    df['htf_rsi'] = rsi(df['close'], 14).fillna(50)
    return df[['time','ema_trend','trend_dir','trend_strength','htf_rsi']]

# ── Reaction filters ────────────────────────────────────────────────────────
def f_bb_touch(df):
    long  = df['low']  <= df['bb_lo']
    short = df['high'] >= df['bb_up']
    return pd.Series(np.where(long,1,np.where(short,-1,0)), index=df.index)

def f_ema_pullback(df):
    tol = PULLBACK_TOLERANCE_ATR * df['atr']
    tl = (df['low']  <= df['ema20']+tol) & (df['close']>df['ema20'])
    ts = (df['high'] >= df['ema20']-tol) & (df['close']<df['ema20'])
    return pd.Series(np.where(tl,1,np.where(ts,-1,0)), index=df.index)

def f_rsi_exit(df):
    prev = df['rsi'].shift(1)
    long  = (prev<=RSI_OS) & (df['rsi']>RSI_OS)
    short = (prev>=RSI_OB) & (df['rsi']<RSI_OB)
    return pd.Series(np.where(long,1,np.where(short,-1,0)), index=df.index)

def f_pin_engulf(df):
    rng = df['range']
    bull_pin = (df['lower_wick']/rng >= PIN_BAR_WICK_RATIO) & (df['close']>df['open'])
    bear_pin = (df['upper_wick']/rng >= PIN_BAR_WICK_RATIO) & (df['close']<df['open'])
    po,pc = df['open'].shift(1), df['close'].shift(1)
    bull_eng = (pc<po)&(df['close']>df['open'])&(df['close']>=po)&(df['open']<=pc)
    bear_eng = (pc>po)&(df['close']<df['open'])&(df['close']<=po)&(df['open']>=pc)
    long  = bull_pin | bull_eng
    short = bear_pin | bear_eng
    return pd.Series(np.where(long,1,np.where(short,-1,0)), index=df.index)

def f_rsi_recent(df, memory=10):
    long_fresh  = (df['f_rsi']== 1).rolling(memory).max().fillna(0).astype(bool)
    short_fresh = (df['f_rsi']==-1).rolling(memory).max().fillna(0).astype(bool)
    return pd.Series(np.where(long_fresh,1,np.where(short_fresh,-1,0)), index=df.index)

def f_macd(df):
    h = df['macd_hist']
    return pd.Series(np.where(h>0, 1, np.where(h<0, -1, 0)), index=df.index)

def f_stoch_cross(df):
    k_prev = df['stoch_k'].shift(1); d_prev = df['stoch_d'].shift(1)
    long  = (k_prev < d_prev) & (df['stoch_k'] > df['stoch_d']) & (df['stoch_k'] < 35)
    short = (k_prev > d_prev) & (df['stoch_k'] < df['stoch_d']) & (df['stoch_k'] > 65)
    return pd.Series(np.where(long, 1, np.where(short, -1, 0)), index=df.index)

def f_volume_spike(df, mult=1.4):
    if 'volume' not in df.columns or df['volume'].sum() == 0:
        return pd.Series(0, index=df.index)
    vm  = df['volume'].rolling(50, min_periods=10).median()
    spike = df['volume'] >= mult * vm
    long_s  = spike & (df['close'] > df['open'])
    short_s = spike & (df['close'] < df['open'])
    return pd.Series(np.where(long_s, 1, np.where(short_s, -1, 0)), index=df.index)

print('indicators + filters defined.')

indicators + filters defined.


## ۴) Backtest engine + Trade dataclass — کپی دقیق از NB24

In [4]:
@dataclass
class Trade:
    side: int; entry_idx: int; entry_time: object
    entry: float; sl: float; tp: float
    exit_idx: int = -1; exit_time: object = None
    exit: float = 0.0; reason: str = ''; r_multiple: float = 0.0

def structural_sl(df, idx, side):
    lo = max(0, idx-STRUCT_LOOKBACK_BARS); a = df['atr'].iat[idx]
    if side == 1:
        return float(df['low'].iloc[lo:idx].min()) - SL_BUFFER_ATR*a
    return float(df['high'].iloc[lo:idx].max()) + SL_BUFFER_ATR*a

def backtest(df, max_hold_bars: int = MAX_HOLD_BARS):
    """Walk bars; open trade on signal bar+1 OPEN, close on SL/TP/time-stop.

    `max_hold_bars` is per-call so each symbol can read its own
    `max_hold_bars` from config.json (per-pair tuning — see
    `_compare_timestop.py` for the 2022-2026 study).
    Falls back to the module-level `MAX_HOLD_BARS` if config omits it.
    """
    trades = []
    n = len(df); in_trade=False; cur=None
    for i in range(n-1):
        if in_trade:
            hi, lo = df['high'].iat[i], df['low'].iat[i]
            hit_sl = (cur.side==1 and lo<=cur.sl) or (cur.side==-1 and hi>=cur.sl)
            hit_tp = (cur.side==1 and hi>=cur.tp) or (cur.side==-1 and lo<=cur.tp)
            exit_now=False; reason=''; px=0.0
            if hit_sl and hit_tp: exit_now,reason,px = True,'sl',cur.sl
            elif hit_sl:          exit_now,reason,px = True,'sl',cur.sl
            elif hit_tp:          exit_now,reason,px = True,'tp',cur.tp
            elif i-cur.entry_idx >= max_hold_bars:
                exit_now,reason,px = True,'time',float(df['close'].iat[i])
            if exit_now:
                cur.exit_idx=i; cur.exit_time=df['time'].iat[i]
                cur.exit=px; cur.reason=reason
                r_unit = abs(cur.entry-cur.sl)
                cur.r_multiple = ((cur.exit-cur.entry)*cur.side)/r_unit if r_unit>0 else 0
                trades.append(cur); in_trade=False; cur=None
        sig = int(df['signal'].iat[i])
        if (not in_trade or not ONE_TRADE_AT_A_TIME) and sig != 0:
            ei = i+1; ep = float(df['open'].iat[ei])
            sl = structural_sl(df, ei, sig)
            r = abs(ep-sl)
            if r<=0 or r>5*df['atr'].iat[ei]: continue
            tp = ep + RR*r*sig
            cur = Trade(side=sig, entry_idx=ei, entry_time=df['time'].iat[ei],
                        entry=ep, sl=sl, tp=tp)
            in_trade=True
    # Open trade at the end of the window: leave it as floating (not in trades list)
    return trades, cur if in_trade else None

def build_signals(m5, h1, d1, cfg):
    m5 = add_m5_features(m5)
    h1t = add_htf_trend(h1, EMA_TREND_H1)
    d1t = add_htf_trend(d1, EMA_TREND_D1)
    m5 = pd.merge_asof(m5.sort_values('time'),
                       h1t.rename(columns={'ema_trend':'h1_ema','trend_dir':'h1_trend',
                                            'trend_strength':'h1_strength','htf_rsi':'h1_rsi'}),
                       on='time', direction='backward')
    m5 = pd.merge_asof(m5,
                       d1t.rename(columns={'ema_trend':'d1_ema','trend_dir':'d1_trend',
                                            'trend_strength':'d1_strength','htf_rsi':'d1_rsi'}),
                       on='time', direction='backward')
    same = m5['h1_trend']==m5['d1_trend']; nz = m5['h1_trend']!=0
    m5['trend_dir'] = np.where(same & nz, m5['h1_trend'], 0).astype(int)
    m5['htf_strong'] = m5['h1_strength'] >= cfg.get('htf_strength_min', 0.0)
    sh, eh = cfg['session']
    ny_h = (m5['time'].dt.hour - BROKER_TO_NY_H) % 24
    m5['in_session'] = (ny_h>=sh) & (ny_h<eh)
    m5['f_bb']     = f_bb_touch(m5)
    m5['f_ema']    = f_ema_pullback(m5)
    m5['f_rsi']    = f_rsi_exit(m5)
    m5['f_candle'] = f_pin_engulf(m5)
    m5['f_rsiR']   = f_rsi_recent(m5, memory=cfg.get('rsi_memory', 10))
    m5['f_macd']   = f_macd(m5)
    m5['f_stoch']  = f_stoch_cross(m5)
    m5['f_vol']    = f_volume_spike(m5, mult=cfg.get('vol_spike_mult', 1.4))
    atr_mult = cfg.get('atr_min_mult', 0.0)
    if atr_mult > 0:
        atr_med = m5['atr'].rolling(500, min_periods=50).median()
        m5['atr_ok'] = (m5['atr'] >= atr_mult * atr_med).fillna(False)
    else:
        m5['atr_ok'] = True
    adx_min = cfg.get('adx_min', 0.0)
    m5['adx_ok'] = m5['adx'] >= adx_min if adx_min > 0 else True
    if cfg.get('require_h1_rsi_align', False):
        m5['h1_rsi_long_ok']  = m5['h1_rsi'] > 50
        m5['h1_rsi_short_ok'] = m5['h1_rsi'] < 50
    else:
        m5['h1_rsi_long_ok']  = True
        m5['h1_rsi_short_ok'] = True
    if cfg.get('require_macd_align', False):
        m5['macd_long_ok']  = m5['f_macd'] ==  1
        m5['macd_short_ok'] = m5['f_macd'] == -1
    else:
        m5['macd_long_ok']  = True
        m5['macd_short_ok'] = True
    base_long  = m5['in_session'] & m5['atr_ok'] & m5['htf_strong'] & m5['adx_ok'] & m5['h1_rsi_long_ok']  & m5['macd_long_ok']
    base_short = m5['in_session'] & m5['atr_ok'] & m5['htf_strong'] & m5['adx_ok'] & m5['h1_rsi_short_ok'] & m5['macd_short_ok']
    if cfg['mode'] == 'RSI-gated':
        rl=(m5['f_rsiR']==1); rs=(m5['f_rsiR']==-1)
        conf = m5[cfg['confirms']].values
        cl=(conf==1).any(axis=1); cs=(conf==-1).any(axis=1)
        cl_long  = (m5['trend_dir']==1)  & base_long  & rl & cl
        cl_short = (m5['trend_dir']==-1) & base_short & rs & cs
    elif cfg['mode'] == 'RSI-gated-AND':
        rl=(m5['f_rsiR']==1); rs=(m5['f_rsiR']==-1)
        conf = m5[cfg['confirms']].values
        cl=(conf==1).all(axis=1); cs=(conf==-1).all(axis=1)
        cl_long  = (m5['trend_dir']==1)  & base_long  & rl & cl
        cl_short = (m5['trend_dir']==-1) & base_short & rs & cs
    elif cfg['mode'] == 'OR':
        mr = cfg.get('min_reactions', 1)
        f  = m5[cfg['confirms']].values
        lv = (f==1).sum(axis=1); sv = (f==-1).sum(axis=1)
        cl_long  = (m5['trend_dir']==1)  & base_long  & (lv>=mr)
        cl_short = (m5['trend_dir']==-1) & base_short & (sv>=mr)
    else:
        raise ValueError(cfg['mode'])
    m5['signal'] = np.where(cl_long,1,np.where(cl_short,-1,0))
    return m5

print('engine ready.')

engine ready.


## ۵) Loader + اجرای بک‌تست برای هر نماد در بازه

هر نماد:
1. M5/H1/D1 را از `(BACKTEST_FROM - WARMUP_DAYS)` تا `BACKTEST_TO` لود می‌کنیم (warmup لازم برای اندیکاتورها)
2. کانفیگ live را از `notebooks/results/multi_symbol_scalper/<sym>/config.json` می‌خوانیم
3. سیگنال‌ها را با موتور NB24 می‌سازیم
4. **فقط ترید‌هایی که `entry_time` در بازه [BACKTEST_FROM, BACKTEST_TO) قرار دارد** را نگه می‌داریم

In [5]:
def load_ohlcv(symbol: str, tf: str, t_from: pd.Timestamp, t_to: pd.Timestamp) -> pd.DataFrame:
    path = DATA_DIR / symbol / tf / 'ohlcv.csv'
    df = pd.read_csv(path)
    df['time'] = pd.to_datetime(df['time']).dt.tz_localize(None)
    df = df.sort_values('time').reset_index(drop=True)
    keep = ['time','open','high','low','close','tick_volume']
    df = df[[c for c in keep if c in df.columns]].copy()
    df.rename(columns={'tick_volume':'volume'}, inplace=True)
    if 'volume' not in df.columns:
        df['volume'] = 0
    df = df[(df['time'] >= t_from) & (df['time'] < t_to)].reset_index(drop=True)
    return df

def load_cfg(sym: str) -> dict:
    p = RESULTS_DIR / sym / 'config.json'
    cfg = json.loads(p.read_text(encoding='utf-8'))
    # session تو JSON معمولاً list هست؛ متن backtest از index 0/1 می‌گیره
    if isinstance(cfg.get('session'), list):
        cfg['session'] = tuple(cfg['session'])
    return cfg

def trade_row(t: Trade, symbol: str) -> dict:
    row = {
        'symbol':     symbol,
        'entry_time': t.entry_time,
        'exit_time':  t.exit_time,
        'side':       'BUY' if t.side==1 else 'SELL',
        'entry':      round(t.entry, 5),
        'sl':         round(t.sl, 5),
        'tp':         round(t.tp, 5),
        'exit':       round(t.exit, 5) if t.exit else None,
        'reason':     t.reason,           # 'tp' | 'sl' | 'time' | '' (open)
        'R':          round(t.r_multiple, 3),
        'hold_bars':  (t.exit_idx - t.entry_idx) if t.exit_idx >= 0 else None,
    }
    if symbol in SYMBOLS_CFG and t.exit:
        ps = SYMBOLS_CFG[symbol]['pip_size']
        pv = SYMBOLS_CFG[symbol]['pip_value_usd_per_002lot']
        sp = SYMBOLS_CFG[symbol]['spread_pips']
        pip_move = (t.exit - t.entry) * t.side / ps
        row['gross_$_at_0.02lot'] = round(pip_move * pv, 2)
        row['fee_$']    = round(sp * pv, 2)
        row['net_$_at_0.02lot'] = round(row['gross_$_at_0.02lot'] - row['fee_$'], 2)
    return row

t_from_user = pd.Timestamp(BACKTEST_FROM)
t_to_user   = pd.Timestamp(BACKTEST_TO)
t_from_load = t_from_user - pd.Timedelta(days=WARMUP_DAYS)
t_to_load   = t_to_user

all_rows = []
open_rows = []
summary_rows = []

for sym in BASKET:
    try:
        cfg = load_cfg(sym)
        m5 = load_ohlcv(sym, 'M5', t_from_load, t_to_load)
        h1 = load_ohlcv(sym, 'H1', t_from_load, t_to_load)
        d1 = load_ohlcv(sym, 'D1', t_from_load, t_to_load)
    except FileNotFoundError as e:
        print(f'  {sym:<8s}  SKIP — missing file: {e.filename}')
        continue
    if len(m5) < 200 or len(h1) < 50 or len(d1) < 5:
        print(f'  {sym:<8s}  SKIP — too little data (M5={len(m5)}, H1={len(h1)}, D1={len(d1)})')
        continue
    m5 = build_signals(m5, h1, d1, cfg)
    # Per-symbol time-stop from config.json (fallback = module-level MAX_HOLD_BARS).
    # Per-pair tuning from _compare_timestop.py: 96 for fast-mean-reverters,
    # 288 for XAUUSD / USDMXN (where short stops chop big winners).
    mhb = int(cfg.get('max_hold_bars', MAX_HOLD_BARS))
    trades, open_trade = backtest(m5, max_hold_bars=mhb)
    # Keep only trades whose ENTRY is inside the requested window
    kept = [t for t in trades if (t.entry_time >= t_from_user) and (t.entry_time < t_to_user)]
    for t in kept:
        all_rows.append(trade_row(t, sym))
    # Trade still open at end of window
    if open_trade is not None and open_trade.entry_time >= t_from_user and open_trade.entry_time < t_to_user:
        open_rows.append(trade_row(open_trade, sym))
    wins = sum(1 for t in kept if t.r_multiple > 0)
    summary_rows.append({
        'symbol':       sym,
        'max_hold_bars': mhb,
        'trades':       len(kept),
        'wins':         wins,
        'losses':       len(kept) - wins,
        'win_rate_%':   round(wins/len(kept)*100, 1) if kept else 0.0,
        'sum_R':        round(sum(t.r_multiple for t in kept), 3),
        'sum_net_$_at_0.02lot': round(sum(
            (((t.exit-t.entry)*t.side/SYMBOLS_CFG[sym]['pip_size'])*SYMBOLS_CFG[sym]['pip_value_usd_per_002lot']
             - SYMBOLS_CFG[sym]['spread_pips']*SYMBOLS_CFG[sym]['pip_value_usd_per_002lot'])
            for t in kept if t.exit), 2),
        'open_at_end':  1 if open_trade is not None and open_trade.entry_time >= t_from_user else 0,
    })
    print(f'  {sym:<8s}  mhb={mhb:>3d}  trades={len(kept):3d}  wins={wins:3d}  WR={(wins/len(kept)*100 if kept else 0):5.1f}%  '
          f'sumR={sum(t.r_multiple for t in kept):+6.2f}')

trades_df = pd.DataFrame(all_rows).sort_values('entry_time').reset_index(drop=True) if all_rows else pd.DataFrame()
open_df   = pd.DataFrame(open_rows) if open_rows else pd.DataFrame()
summary_df = pd.DataFrame(summary_rows)
print(f'\nTotal trades in window: {len(trades_df)}   |   still-open at end: {len(open_df)}')

  GBPUSD    mhb= 96  trades=  4  wins=  1  WR= 25.0%  sumR= -1.92


  XAUUSD    mhb=288  trades=  3  wins=  1  WR= 33.3%  sumR= +0.00


  GBPCAD    mhb= 96  trades=  2  wins=  1  WR= 50.0%  sumR= +1.00


  USDMXN    mhb=288  trades=  3  wins=  1  WR= 33.3%  sumR= +0.00


  EURJPY    mhb= 96  trades=  0  wins=  0  WR=  0.0%  sumR= +0.00


  EURCAD    mhb= 96  trades=  3  wins=  1  WR= 33.3%  sumR= +0.00


  AUDCAD    mhb= 96  trades=  0  wins=  0  WR=  0.0%  sumR= +0.00

Total trades in window: 15   |   still-open at end: 0


## ۶) نمایش لیست ترید‌ها و خلاصه

In [6]:
print('=== summary per symbol ===')
print(summary_df.to_string(index=False))
print()
if not trades_df.empty:
    print('=== all closed trades (sorted by entry_time) ===')
    print(trades_df.to_string(index=False))
if not open_df.empty:
    print('\n=== trades still open at end of window ===')
    print(open_df.to_string(index=False))

=== summary per symbol ===
symbol  max_hold_bars  trades  wins  losses  win_rate_%    sum_R  sum_net_$_at_0.02lot  open_at_end
GBPUSD             96       4     1       3    25.00000 -1.91900              -4.56000            0
XAUUSD            288       3     1       2    33.30000  0.00000             -17.53000            0
GBPCAD             96       2     1       1    50.00000  1.00000              -0.00000            0
USDMXN            288       3     1       2    33.30000  0.00000              -1.32000            0
EURJPY             96       0     0       0     0.00000  0.00000               0.00000            0
EURCAD             96       3     1       2    33.30000  0.00000              -3.60000            0
AUDCAD             96       0     0       0     0.00000  0.00000               0.00000            0

=== all closed trades (sorted by entry_time) ===
symbol          entry_time           exit_time side      entry         sl         tp       exit reason        R  hold_bars 

## ۷) ذخیره به CSV

In [7]:
tag = f'{BACKTEST_FROM}_{BACKTEST_TO}'.replace('-', '')
out_trades  = OUT_DIR / f'replay_trades_{tag}.csv'
out_summary = OUT_DIR / f'replay_summary_{tag}.csv'
out_open    = OUT_DIR / f'replay_open_{tag}.csv'

if not trades_df.empty:
    trades_df.to_csv(out_trades, index=False)
    print(f'saved {len(trades_df):3d} closed trades  → {out_trades}')
else:
    print('no closed trades in window')

summary_df.to_csv(out_summary, index=False)
print(f'saved summary               → {out_summary}')

if not open_df.empty:
    open_df.to_csv(out_open, index=False)
    print(f'saved {len(open_df):3d} open trades       → {out_open}')

print()
print('برای مقایسه با live: ترید‌های `event=market_order_placed` و `position_closed_detected`')
print('در `logs/<SYM>-YYYY-MM-DD.json` را با ستون‌های entry_time/entry/sl/tp مقایسه کن.')

saved  15 closed trades  → D:\bot\ema-1d trend\ema-h1trend\notebooks\data\replay_trades_20260527_20260530.csv
saved summary               → D:\bot\ema-1d trend\ema-h1trend\notebooks\data\replay_summary_20260527_20260530.csv

برای مقایسه با live: ترید‌های `event=market_order_placed` و `position_closed_detected`
در `logs/<SYM>-YYYY-MM-DD.json` را با ستون‌های entry_time/entry/sl/tp مقایسه کن.


## ۸) مقایسه: همان بازه‌ی زمانی بدون فیلتر سشن نیویورک

همان موتور را روی همان بازه دوباره اجرا می‌کنیم — فقط `session` کانفیگ هر نماد را به `(0, 24)` تغییر می‌دهیم: یعنی **در همه‌ی ساعات روز** ترید مجاز است. سایر گیت‌ها (روند ساعت بزرگ، روند روزانه، RSI ساعت بزرگ، MACD، ...) دست‌نخورده می‌مانند.


In [8]:
all_rows_24h = []
summary_rows_24h = []

for sym in BASKET:
    try:
        cfg = dict(load_cfg(sym))
        cfg['session'] = (0, 24)   # ← تنها تفاوت با سل ۱۰
        m5 = load_ohlcv(sym, 'M5', t_from_load, t_to_load)
        h1 = load_ohlcv(sym, 'H1', t_from_load, t_to_load)
        d1 = load_ohlcv(sym, 'D1', t_from_load, t_to_load)
    except FileNotFoundError:
        continue
    if len(m5) < 200 or len(h1) < 50 or len(d1) < 5:
        continue
    m5 = build_signals(m5, h1, d1, cfg)
    mhb = int(cfg.get('max_hold_bars', MAX_HOLD_BARS))
    trades, _ = backtest(m5, max_hold_bars=mhb)
    kept = [t for t in trades if (t.entry_time >= t_from_user) and (t.entry_time < t_to_user)]
    for t in kept:
        row = trade_row(t, sym); row['session'] = 'all_day'
        all_rows_24h.append(row)
    wins = sum(1 for t in kept if t.r_multiple > 0)
    net = round(sum(
        (((t.exit-t.entry)*t.side/SYMBOLS_CFG[sym]['pip_size'])*SYMBOLS_CFG[sym]['pip_value_usd_per_002lot']
         - SYMBOLS_CFG[sym]['spread_pips']*SYMBOLS_CFG[sym]['pip_value_usd_per_002lot'])
        for t in kept if t.exit), 2)
    summary_rows_24h.append({
        'symbol':       sym,
        'max_hold_bars': mhb,
        'trades':       len(kept),
        'wins':         wins,
        'losses':       len(kept) - wins,
        'win_rate_%':   round(wins/len(kept)*100, 1) if kept else 0.0,
        'sum_R':        round(sum(t.r_multiple for t in kept), 3),
        'sum_net_$_at_0.02lot': net,
    })
    print(f'  {sym:<8s}  mhb={mhb:>3d}  trades={len(kept):4d}  wins={wins:3d}  WR={(wins/len(kept)*100 if kept else 0):5.1f}%  '
          f'sumR={sum(t.r_multiple for t in kept):+6.2f}  net=${net:+7.2f}')

summary_24h_df = pd.DataFrame(summary_rows_24h)
trades_24h_df  = pd.DataFrame(all_rows_24h).sort_values('entry_time').reset_index(drop=True) if all_rows_24h else pd.DataFrame()
print(f'\nمجموع ترید بدون فیلتر سشن: {len(trades_24h_df)}')


  GBPUSD    mhb= 96  trades=   8  wins=  3  WR= 37.5%  sumR= +0.08  net=$  -2.44


  XAUUSD    mhb=288  trades=   8  wins=  5  WR= 62.5%  sumR= +7.00  net=$+115.64


  GBPCAD    mhb= 96  trades=   2  wins=  2  WR=100.0%  sumR= +2.36  net=$  +2.45


  USDMXN    mhb=288  trades=   3  wins=  1  WR= 33.3%  sumR= +0.00  net=$  -1.56


  EURJPY    mhb= 96  trades=   2  wins=  2  WR=100.0%  sumR= +2.06  net=$  +0.79


  EURCAD    mhb= 96  trades=   7  wins=  4  WR= 57.1%  sumR= +4.31  net=$  -1.35


  AUDCAD    mhb= 96  trades=   0  wins=  0  WR=  0.0%  sumR= +0.00  net=$  +0.00

مجموع ترید بدون فیلتر سشن: 30


## ۹) جدول مقایسه: سشن نیویورک vs کل روز


In [9]:
# سمت چپ = نسخه‌ی سشن نیویورک (سل ۱۰)  /  سمت راست = کل روز (سل بالا)
left  = summary_df.set_index('symbol').add_suffix('_ny')
right = summary_24h_df.set_index('symbol').add_suffix('_24h')
cmp = left.join(right, how='outer').reset_index()

cols = ['symbol','trades_ny','trades_24h',
        'win_rate_%_ny','win_rate_%_24h',
        'sum_R_ny','sum_R_24h',
        'sum_net_$_at_0.02lot_ny','sum_net_$_at_0.02lot_24h']
view = cmp[[c for c in cols if c in cmp.columns]].copy()
view['delta_trades']   = view['trades_24h']        - view['trades_ny']
view['delta_WR']       = view['win_rate_%_24h']    - view['win_rate_%_ny']
view['delta_sumR']     = view['sum_R_24h']         - view['sum_R_ny']
view['delta_net_$']    = view['sum_net_$_at_0.02lot_24h'] - view['sum_net_$_at_0.02lot_ny']

print('=== مقایسه‌ی per-symbol (NY = سشن فعلی، 24h = کل روز) ===')
print(view.to_string(index=False))

print('\n=== مجموع کل سبد ===')
tot_ny = {
    'trades':   int(summary_df['trades'].sum()),
    'wins':     int(summary_df['wins'].sum()),
    'sum_R':    round(summary_df['sum_R'].sum(), 2),
    'net_$':    round(summary_df['sum_net_$_at_0.02lot'].sum(), 2),
}
tot_24 = {
    'trades':   int(summary_24h_df['trades'].sum()),
    'wins':     int(summary_24h_df['wins'].sum()),
    'sum_R':    round(summary_24h_df['sum_R'].sum(), 2),
    'net_$':    round(summary_24h_df['sum_net_$_at_0.02lot'].sum(), 2),
}
tot_ny['WR'] = round(tot_ny['wins']/tot_ny['trades']*100, 1) if tot_ny['trades'] else 0.0
tot_24['WR'] = round(tot_24['wins']/tot_24['trades']*100, 1) if tot_24['trades'] else 0.0
print(f'  NY-session:   trades={tot_ny["trades"]:4d}  WR={tot_ny["WR"]:5.1f}%  sumR={tot_ny["sum_R"]:+7.2f}  net=${tot_ny["net_$"]:+8.2f}')
print(f'  all-day:      trades={tot_24["trades"]:4d}  WR={tot_24["WR"]:5.1f}%  sumR={tot_24["sum_R"]:+7.2f}  net=${tot_24["net_$"]:+8.2f}')
print(f'  delta(24-NY): Δtrades={tot_24["trades"]-tot_ny["trades"]:+4d}  ΔWR={tot_24["WR"]-tot_ny["WR"]:+5.1f}%  '
      f'ΔsumR={tot_24["sum_R"]-tot_ny["sum_R"]:+6.2f}  Δnet=${tot_24["net_$"]-tot_ny["net_$"]:+8.2f}')

out_cmp = OUT_DIR / f'replay_session_compare_{tag}.csv'
view.to_csv(out_cmp, index=False)
print(f'\nذخیره شد → {out_cmp}')


=== مقایسه‌ی per-symbol (NY = سشن فعلی، 24h = کل روز) ===
symbol  trades_ny  trades_24h  win_rate_%_ny  win_rate_%_24h  sum_R_ny  sum_R_24h  sum_net_$_at_0.02lot_ny  sum_net_$_at_0.02lot_24h  delta_trades  delta_WR  delta_sumR  delta_net_$
AUDCAD          0           0        0.00000         0.00000   0.00000    0.00000                  0.00000                   0.00000             0   0.00000     0.00000      0.00000
EURCAD          3           7       33.30000        57.10000   0.00000    4.30600                 -3.60000                  -1.35000             4  23.80000     4.30600      2.25000
EURJPY          0           2        0.00000       100.00000   0.00000    2.06400                  0.00000                   0.79000             2 100.00000     2.06400      0.79000
GBPCAD          2           2       50.00000       100.00000   1.00000    2.35900                 -0.00000                   2.45000             0  50.00000     1.35900      2.45000
GBPUSD          4           8   

## ۱۰) مقایسه‌ی ماه‌به‌ماه — سشن نیویورک vs کل روز

برای فهم اینکه آیا حذف فیلتر سشن در **هر ماه** سودده‌تر است یا فقط در میانگین کل، دو جدول می‌سازیم:

- جدول الف — تجمیع ماهانه‌ی سبد (مجموع همه‌ی نمادها)
- جدول ب — تجمیع ماهانه به تفکیک هر نماد

هر سطر: ماه + سناریو + تعداد ترید + درصد برد + جمع ضریب R + سود خالص دلاری.


In [10]:
# Build a unified long-format trades table: NY session + all_day, both labelled.
ny  = trades_df.copy()    if not trades_df.empty   else pd.DataFrame()
h24 = trades_24h_df.copy() if not trades_24h_df.empty else pd.DataFrame()
if not ny.empty:
    ny['scenario']  = 'NY'
if not h24.empty:
    h24['scenario'] = '24h'
all_t = pd.concat([ny, h24], ignore_index=True) if (not ny.empty or not h24.empty) else pd.DataFrame()

if all_t.empty:
    print('هیچ ترید در dataframe ها نیست — ابتدا سل ۱۰ و سل ۸ را اجرا کنید.')
else:
    all_t['entry_time'] = pd.to_datetime(all_t['entry_time'])
    all_t['month']      = all_t['entry_time'].dt.strftime('%Y-%m')
    # net column may be missing if a trade has no exit (still-open is excluded by the filter above)
    net_col = 'net_$_at_0.02lot'
    if net_col not in all_t.columns:
        all_t[net_col] = 0.0
    else:
        all_t[net_col] = all_t[net_col].fillna(0.0)
    all_t['is_win'] = (all_t['R'] > 0).astype(int)

    # ─── جدول الف: تجمیع ماهانه‌ی سبد (همه‌ی نمادها در هر ماه) ──────────
    by_month = (all_t.groupby(['month','scenario'], as_index=False)
                     .agg(trades=('R','count'),
                          wins=('is_win','sum'),
                          sum_R=('R','sum'),
                          net_usd=(net_col,'sum')))
    by_month['win_rate_%'] = (by_month['wins']/by_month['trades']*100).round(1)
    by_month = by_month[['month','scenario','trades','wins','win_rate_%','sum_R','net_usd']]

    # Pivot to side-by-side per month so eye can compare quickly
    pvt = (by_month.set_index(['month','scenario'])[['trades','win_rate_%','sum_R','net_usd']]
                   .unstack('scenario'))
    # Flatten multiindex cols
    pvt.columns = [f'{a}_{b}' for a, b in pvt.columns]
    pvt = pvt.reset_index().fillna(0)
    # Delta columns (24h minus NY)
    for k in ('trades','win_rate_%','sum_R','net_usd'):
        col_ny  = f'{k}_NY'
        col_24  = f'{k}_24h'
        if col_ny in pvt.columns and col_24 in pvt.columns:
            pvt[f'delta_{k}'] = (pvt[col_24] - pvt[col_ny]).round(2)
    # Round display columns
    for c in pvt.columns:
        if c == 'month': continue
        try: pvt[c] = pvt[c].round(2)
        except: pass

    print('=== جدول الف: تجمیع ماهانه (همه‌ی سبد در یک ردیف هر ماه) ===')
    print(pvt.to_string(index=False))

    # ─── جدول ب: تجمیع ماهانه به تفکیک نماد ────────────────────────────
    by_sym = (all_t.groupby(['month','symbol','scenario'], as_index=False)
                   .agg(trades=('R','count'),
                        wins=('is_win','sum'),
                        sum_R=('R','sum'),
                        net_usd=(net_col,'sum')))
    by_sym['win_rate_%'] = (by_sym['wins']/by_sym['trades']*100).round(1)
    by_sym = by_sym[['month','symbol','scenario','trades','win_rate_%','sum_R','net_usd']].sort_values(['month','symbol','scenario'])
    print()
    print('=== جدول ب: تجمیع ماهانه به تفکیک نماد ===')
    print(by_sym.to_string(index=False))

    # Save both to CSV
    out_a = OUT_DIR / f'replay_monthly_compare_{tag}.csv'
    out_b = OUT_DIR / f'replay_monthly_per_symbol_{tag}.csv'
    pvt.to_csv(out_a, index=False)
    by_sym.to_csv(out_b, index=False)
    print(f'\nذخیره شد → {out_a}')
    print(f'ذخیره شد → {out_b}')


=== جدول الف: تجمیع ماهانه (همه‌ی سبد در یک ردیف هر ماه) ===
  month  trades_24h  trades_NY  win_rate_%_24h  win_rate_%_NY  sum_R_24h  sum_R_NY  net_usd_24h  net_usd_NY  delta_trades  delta_win_rate_%  delta_sum_R  delta_net_usd
2026-05          30         15        56.70000       33.30000   15.81000  -0.92000    113.49000   -27.01000            15          23.40000     16.73000      140.50000

=== جدول ب: تجمیع ماهانه به تفکیک نماد ===
  month symbol scenario  trades  win_rate_%    sum_R   net_usd
2026-05 EURCAD      24h       7    57.10000  4.30600  -1.39000
2026-05 EURCAD       NY       3    33.30000  0.00000  -3.61000
2026-05 EURJPY      24h       2   100.00000  2.06400   0.80000
2026-05 GBPCAD      24h       2   100.00000  2.35900   2.46000
2026-05 GBPCAD       NY       2    50.00000  1.00000   0.00000
2026-05 GBPUSD      24h       8    37.50000  0.08100  -2.45000
2026-05 GBPUSD       NY       4    25.00000 -1.91900  -4.55000
2026-05 USDMXN      24h       3    33.30000  0.00000  -